# Chapter 21 — Convolutional Neural Networks I: Convolution, Pooling & CNN Structure

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part3_neural_networks/ch21_convolutional_neural_networks_i/ch21_notebook.ipynb)

Companion notebook for **Chapter 21** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- Convolution slides a small kernel across the input and computes a dot product at each position.
- Sparse connectivity plus weight sharing gives a CNN far fewer parameters than an equivalent fully connected layer.
- Pooling downsamples a feature map and has no learnable parameters at all.
- Feature-map size follows W2 = (W1 - F + 2P)/S + 1, so filter size, stride and padding fully determine the output shape.
- Convolution and pooling together give the network translation invariance.

## Example 1: 1D Convolution from Scratch

This works through a small 1-D convolution by hand. We slide a length-4 kernel across a 10-element signal with stride 2, doing the shift-and-multiply (dot product) at each stop. The output is the feature map [-0.6, 3.2, 2.8, -0.5], and we confirm the output-size formula. Note there is no bias and no activation - that is the raw convolution.

In [1]:
# Chapter 21 - Example 1: 1D Convolution from scratch
import numpy as np

def conv1d(data, kernel, stride):
    K = len(kernel)
    out = []
    positions = []
    for start in range(0, len(data) - K + 1, stride):
        window = data[start:start+K]
        val = np.dot(window, kernel)
        out.append(round(float(val), 4))
        positions.append(start)
    return out, positions

# A small worked example
data   = np.array([2, 8, 6, 4, 7, 3, 5, 2, 1, 4], dtype=float)
kernel = np.array([0.2, -0.5, 0.7, -0.3], dtype=float)
stride = 2

print("Input data  :", data.astype(int).tolist())
print("Kernel      :", kernel.tolist())
print("Stride      :", stride, "| Kernel size:", len(kernel))
print()

features, pos = conv1d(data, kernel, stride)
print("Step-by-step (shift-and-multiply):")
for f, p in zip(features, pos):
    window = data[p:p+len(kernel)].astype(int).tolist()
    terms  = " + ".join(f"{k}*{w}" for k, w in zip(kernel.tolist(), window))
    print(f"  pos {p}: {terms} = {f}")
print()
print("Feature map extracted:", features)
print("Expected by hand     : [-0.6, 3.2, 2.8, -0.5]")

# Output length formula
W1 = len(data); F = len(kernel); S = stride
W2 = (W1 - F)//S + 1
print(f"\nOutput size formula: (W1 - F)/S + 1 = ({W1} - {F})/{S} + 1 = {W2}")
print("Note: convolution layer has NO bias and NO activation in this raw operation.")

Input data  : [2, 8, 6, 4, 7, 3, 5, 2, 1, 4]
Kernel      : [0.2, -0.5, 0.7, -0.3]
Stride      : 2 | Kernel size: 4

Step-by-step (shift-and-multiply):
  pos 0: 0.2*2 + -0.5*8 + 0.7*6 + -0.3*4 = -0.6
  pos 2: 0.2*6 + -0.5*4 + 0.7*7 + -0.3*3 = 3.2
  pos 4: 0.2*7 + -0.5*3 + 0.7*5 + -0.3*2 = 2.8
  pos 6: 0.2*5 + -0.5*2 + 0.7*1 + -0.3*4 = -0.5

Feature map extracted: [-0.6, 3.2, 2.8, -0.5]
Expected by hand     : [-0.6, 3.2, 2.8, -0.5]

Output size formula: (W1 - F)/S + 1 = (10 - 4)/2 + 1 = 4
Note: convolution layer has NO bias and NO activation in this raw operation.


## Example 2: 2D Convolution + Max/Mean Pooling

Here we build 2-D convolution and both pooling types from scratch, using a small 4x4 image and a 2x2 kernel. You can check every output value, and the max-pooled and mean-pooled values, by hand. This shows how a feature map is built and then downsampled - and that pooling has no learnable parameters.

In [2]:
# Chapter 21 - Example 2: 2D Convolution + Max/Mean Pooling from scratch
import numpy as np

def conv2d(img, kernel, stride=1):
    kh, kw = kernel.shape
    H, W = img.shape
    out_h = (H - kh)//stride + 1
    out_w = (W - kw)//stride + 1
    out = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            region = img[i*stride:i*stride+kh, j*stride:j*stride+kw]
            out[i, j] = np.sum(region * kernel)
    return out

def pool2d(fmap, size=2, stride=2, mode='max'):
    H, W = fmap.shape
    out_h = (H - size)//stride + 1
    out_w = (W - size)//stride + 1
    out = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            region = fmap[i*stride:i*stride+size, j*stride:j*stride+size]
            out[i, j] = region.max() if mode == 'max' else region.mean()
    return out

# Input (4x4), 2x2 kernel, stride 1
img = np.array([[5, 9, 3, 1],
                [7, 2, 6, 4],
                [1, 8, 5, 2],
                [3, 0, 7, 9]], dtype=float)
kernel = np.array([[0.2, -0.5],
                   [0.7, -0.3]], dtype=float)

print("Input image (4x4):")
print(img.astype(int))
print("\n2x2 Kernel:")
print(kernel)

fmap = conv2d(img, kernel, stride=1)
print("\nFeature map after 2D convolution (stride=1, 3x3):")
print(np.round(fmap, 2))

# Now pooling on a sample 4x4 feature map
feat = np.array([[ 0.8, -0.1,  3.1, -1.6],
                 [-1.3,  1.5,  2.1, -0.6],
                 [-1.7, -3.0,  2.2,  0.4],
                 [ 2.6,  0.8, -1.5,  2.8]], dtype=float)
print("\n--- Pooling demo on a 4x4 feature map ---")
print("Features before pooling:")
print(feat)
print("\nMax pooling (2x2, stride 2):")
print(np.round(pool2d(feat, 2, 2, 'max'), 3))
print("Expected: [[1.5, 3.1], [2.6, 2.8]]")
print("\nMean pooling (2x2, stride 2):")
print(np.round(pool2d(feat, 2, 2, 'mean'), 3))
print("Expected: [[0.225, 0.75], [-0.325, 0.975]]")
print("\nPooling kernels have NO learnable parameters.")

Input image (4x4):
[[5 9 3 1]
 [7 2 6 4]
 [1 8 5 2]
 [3 0 7 9]]

2x2 Kernel:
[[ 0.2 -0.5]
 [ 0.7 -0.3]]

Feature map after 2D convolution (stride=1, 3x3):
[[ 0.8 -0.1  3.1]
 [-1.3  1.5  2.1]
 [-1.7 -3.   2.2]]

--- Pooling demo on a 4x4 feature map ---
Features before pooling:
[[ 0.8 -0.1  3.1 -1.6]
 [-1.3  1.5  2.1 -0.6]
 [-1.7 -3.   2.2  0.4]
 [ 2.6  0.8 -1.5  2.8]]

Max pooling (2x2, stride 2):
[[1.5 3.1]
 [2.6 2.8]]
Expected: [[1.5, 3.1], [2.6, 2.8]]

Mean pooling (2x2, stride 2):
[[ 0.225  0.75 ]
 [-0.325  0.975]]
Expected: [[0.225, 0.75], [-0.325, 0.975]]

Pooling kernels have NO learnable parameters.


## Example 3: Feature-Map Size & MLFFNN vs CNN Parameters

This applies the feature-map size formula to several configurations (including padding) and then drives home why CNNs are efficient: a fully-connected layer on 16 pixels needs 64 weights, while the equivalent convolution needs just the shared kernel weights - a ~94% reduction. The LeNet-5 example shows ~5 million dense weights collapsing to ~156 with weight sharing.

In [3]:
# Chapter 21 - Example 3: Feature-map size + MLFFNN vs CNN parameter comparison
import numpy as np

def feature_map_size(W1, H1, F, S, P=0):
    W2 = (W1 - F + 2*P)//S + 1
    H2 = (H1 - F + 2*P)//S + 1
    return W2, H2

print("=== Feature-Map Size Formula: W2 = (W1 - F + 2P)/S + 1 ===\n")
configs = [
    (4, 4, 2, 2, 0),
    (4, 4, 2, 1, 0),
    (32, 32, 3, 2, 0),
    (32, 32, 4, 3, 0),
    (4, 4, 2, 2, 2),   # with padding
]
print(f"{'Input':>8} {'Filter':>7} {'Stride':>7} {'Pad':>5} {'Output':>8}")
print("-" * 40)
for W1, H1, F, S, P in configs:
    W2, H2 = feature_map_size(W1, H1, F, S, P)
    print(f"{W1}x{H1:<5} {F}x{F:<5} {S:>5} {P:>5} {W2}x{H2}")

print("\n=== MLFFNN vs CNN: Parameter Count (16-pixel input, 4 hidden units) ===\n")
# MLFFNN: fully connected — every input connects to every hidden node
n_input  = 16   # 4x4 image flattened
n_hidden = 4
mlffnn_weights = n_input * n_hidden
print(f"MLFFNN (fully connected):")
print(f"  Connections = {n_input} inputs x {n_hidden} hidden = {mlffnn_weights} weights")

# CNN: 2x2 kernel, stride 2 -> 4 output nodes, but SHARED weights
kernel_size = 2 * 2   # 2x2 kernel
print(f"\nCNN (convolution, 2x2 kernel, stride 2):")
print(f"  4 hidden nodes each use the SAME 2x2 kernel (weight sharing)")
print(f"  Unique weights = {kernel_size} (one shared kernel) + 1 bias = {kernel_size + 1}")
print(f"\nParameter reduction: {mlffnn_weights} -> {kernel_size} "
      f"({(1 - kernel_size/mlffnn_weights)*100:.1f}% fewer weights)")

# LeNet-5 style realistic example
print("\n=== Real example: LeNet-5 first conv layer ===")
print("Input 32x32x1, Conv layer CL1: 6 kernels of 5x5, stride 1")
n_kernels = 6; ksz = 5*5
conv_params = n_kernels * (ksz + 1)  # +1 bias per kernel
W2, H2 = feature_map_size(32, 32, 5, 1, 0)
print(f"  Output feature maps: {W2}x{H2}x{n_kernels}")
print(f"  Learnable parameters: {n_kernels} x (5x5 + 1 bias) = {conv_params}")
fc_equiv = (32*32) * (W2*H2*n_kernels)
print(f"  A fully-connected layer for the same output would need ~{fc_equiv:,} weights")
print(f"  CNN uses {conv_params} -- that's the power of weight sharing + sparse connections!")

=== Feature-Map Size Formula: W2 = (W1 - F + 2P)/S + 1 ===

   Input  Filter  Stride   Pad   Output
----------------------------------------
4x4     2x2         2     0 2x2
4x4     2x2         1     0 3x3
32x32    3x3         2     0 15x15
32x32    4x4         3     0 10x10
4x4     2x2         2     2 4x4

=== MLFFNN vs CNN: Parameter Count (16-pixel input, 4 hidden units) ===

MLFFNN (fully connected):
  Connections = 16 inputs x 4 hidden = 64 weights

CNN (convolution, 2x2 kernel, stride 2):
  4 hidden nodes each use the SAME 2x2 kernel (weight sharing)
  Unique weights = 4 (one shared kernel) + 1 bias = 5

Parameter reduction: 64 -> 4 (93.8% fewer weights)

=== Real example: LeNet-5 first conv layer ===
Input 32x32x1, Conv layer CL1: 6 kernels of 5x5, stride 1
  Output feature maps: 28x28x6
  Learnable parameters: 6 x (5x5 + 1 bias) = 156
  A fully-connected layer for the same output would need ~4,816,896 weights
  CNN uses 156 -- that's the power of weight sharing + sparse connecti